In [2]:
%pip install requests -q

In [3]:
from google.colab import userdata

KEY: str = userdata.get("KOREAN_DICT_KEY")
print(KEY[:4] + "***")

## 이 셀은 클로드의 도움을 받음
## VSCODE가 아닌 구글코랩을 이용하였음

3AD4***


---
## Q1

In [4]:
# (a)
import requests

def search_word(q: str, num: int = 10, start: int = 1) -> dict:
    endpoint = "https://opendict.korean.go.kr/api/search"
    params = {
        "key": KEY,
        "q": q,
        "req_type": "json",
        "num": num,
        "start": start,
        "type1": "word",
    }
    r = requests.get(endpoint, params=params, timeout=10)
    r.raise_for_status()
    return r.json()

search_word 함수로 우리말샘 API의 엔드포인트에 requests.get으로 요청을 보냈다. timeout으로 응답 지연을 방지했고, raise_for_status()로 오류가 있으면 즉시 예외를 발생시켰다.

In [5]:
# (b)
import json

data = search_word("김치")
print(json.dumps(data, ensure_ascii=False, indent=2)[:400])

{
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
          


응답의 최상위 키는 channel이고 그 안에 total, num, item 등이 들어 있다.
ensure_ascii=False를 빼면 사람이 읽기 어려워지게 출력된다.

In [11]:
# (c)

total: int = int(data["channel"]["total"])
n: int = int(data["channel"]["num"])
print(f"총 {total}건, 이 페이지 {n}건")


items: list = data["channel"]["item"]
for item in items[:5]:
    word: str = item.get("word", "")
    sense_list = item.get("sense", [])

    if isinstance(sense_list, list) and sense_list:
        pos: str = sense_list[0].get("pos", "품사 없음") or "품사 없음"
        definition: str = sense_list[0].get("definition", "")
    elif isinstance(sense_list, dict):
        pos = sense_list.get("pos", "품사 없음") or "품사 없음"
        definition = sense_list.get("definition", "")
    else:
        pos = "품사 없음"
        definition = ""

    print(f"{word} ({pos}) --> {definition[:40]}")

총 328건, 이 페이지 10건
김치 (명사) --> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린
김-치 (명사) --> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 
김-치 (명사) --> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南
김치 공장 (품사 없음) --> 김치를 만드는 공장.
김치 보릿고개 (품사 없음) --> 김장철인 가을·겨울과 달리 상대적으로 김치가 부족한 봄여름을 비유적으로 


dict.get으로 pos에 안전하게 접근하여 없거나 빈 문자열이면 품사 없음을 기본값으로 사용한다.

## Q2

In [7]:
import time
from collections import Counter

words: list[str] = [
    "김치", "라면", "만두", "김밥",
    "국수", "떡볶이", "불고기", "비빔밥",
]

results: dict[str, dict] = {}
for w in words:
    results[w] = search_word(w)
    time.sleep(0.3)

In [9]:
# (a)
for w in words:
    total_count: int = int(results[w]["channel"]["total"])
    print(f"{w}: {total_count}건")

김치: 328건
라면: 86건
만두: 89건
김밥: 39건
국수: 227건
떡볶이: 24건
불고기: 38건
비빔밥: 38건


각 검색어에 대해 data["channel"]["total"]에서 전체 결과 수를 가져와 출력한다. 결과 수의 차이를 통해 각 음식 단어가 사전에 얼마나 다양하게 등재되어 있는지 비교할 수 있다.

In [12]:
# (b)
all_items: list = []
for w in words:
    channel = results[w]["channel"]
    item_data = channel.get("item", [])
    if isinstance(item_data, list):
        all_items.extend(item_data)
    elif isinstance(item_data, dict):
        all_items.append(item_data)

def get_pos(item: dict) -> str:
    sense_list = item.get("sense", [])
    if isinstance(sense_list, list) and sense_list:
        return sense_list[0].get("pos") or "(미상)"
    elif isinstance(sense_list, dict):
        return sense_list.get("pos") or "(미상)"
    return "(미상)"

pos_counter: Counter = Counter(
    get_pos(item) for item in all_items
)

print("품사 빈도 상위 3개:")
for pos_label, freq in pos_counter.most_common(3):
    print(f"  {pos_label}: {freq}건")

## pos가 item 아래가 아니라 sense 안에 있어서 전부 "(미상)"으로 카운트되는 문제를 클로드의 도움을 받아 해결하였음

품사 빈도 상위 3개:
  명사: 60건
  (미상): 19건
  어미: 1건


8개 검색어의 결과 항목을 all_items 리스트에 합친 뒤, item.get("pos") or "(미상)"으로 품사가 없거나 빈 문자열인 경우를 `"(미상)"`으로 통일하여 센다. 음식 이름은 대부분 사물,개념을 지칭하는 명사이며, 복합어, 파생어,외래어도 명사로 등재되기 때문에 명사 항목이 많다
